# Bird Eye v33: train a 2-class YOLO11n on RDD2022 and export float16 TFLite

Run on a Colab GPU runtime. Set `RUN_MODE` in cell 1, then run all. Modes: `smoke` (random weights, no dataset, exports a TFLite today for the app), `quick` (10% of data, 3 epochs), `full` (all data, 30 epochs). Every cell is safe to re-run after a disconnect. See `ml/README.md`.

## 1. Config
All settings live here. Cell 1 also mounts Drive, because the model version is stored on Drive: it is written the first time this cell runs and read on every later run, so a resume on another day keeps the same version. Delete `model_version_<RUN_MODE>.json` on Drive to get a new version. For `quick` and `full` the version is `m-<date>-<RUN_MODE>`; an existing file is read as it is and never rewritten.

In [ ]:
import datetime, hashlib, json, os
from pathlib import Path

RUN_MODE = "smoke"  # "smoke", "quick" or "full"
RUN_SETTINGS = {
    "smoke": {"fraction": None, "epochs": 0},  # no dataset, no training
    "quick": {"fraction": 0.1, "epochs": 3},
    "full": {"fraction": 1.0, "epochs": 30},
}
if RUN_MODE not in RUN_SETTINGS:
    raise ValueError(f'RUN_MODE must be "smoke", "quick" or "full", got {RUN_MODE!r}')
SMOKE = RUN_MODE == "smoke"
FRACTION = RUN_SETTINGS[RUN_MODE]["fraction"]
EPOCHS = RUN_SETTINGS[RUN_MODE]["epochs"]

MODEL = "yolo11n.pt"
IMGSZ = 640
SEED = 0
VAL_FRACTION = 0.1
DRIVE_ROOT = "/content/drive/MyDrive/birdeye_ml"
LOCAL_DATA = "/content/rdd2022_yolo"  # fixed absolute path, so resume works across sessions

# Class order is fixed: 0 = pothole, 1 = crack.
CLASS_NAMES = ["pothole", "crack"]
CODE_TO_CLASS = {"D40": 0, "D00": 1, "D10": 1, "D20": 1}
# Every other RDD2022 code is dropped on purpose. An unknown code stops cell 5.
DROP_CODES = {"D01", "D0w0", "D11", "D43", "D44", "D50", "Repair", "Block crack"}
# Drone images are top down; the patrol phone looks forward through a windscreen. China_MotorBike is kept.
EXCLUDE_COUNTRIES = ["China_Drone"]
# Val is whole blocks of consecutive images (by file name), not random images: neighbouring frames can look alike,
# and a random split would put near duplicates in both train and val.
SPLIT_BLOCK = 100
SPLIT_METHOD = f"blocks{SPLIT_BLOCK}"

# Short hash of everything that changes the converted dataset. It names the cache and the split file,
# so a cache built with other settings is never reused.
BUILD_SETTINGS = {
    "code_to_class": CODE_TO_CLASS, "drop_codes": sorted(DROP_CODES), "exclude_countries": sorted(EXCLUDE_COUNTRIES),
    "imgsz": IMGSZ, "seed": SEED, "val_fraction": VAL_FRACTION, "split_method": SPLIT_METHOD,
}
SETTINGS_HASH = hashlib.sha1(json.dumps(BUILD_SETTINGS, sort_keys=True).encode()).hexdigest()[:8]
# One hash names the cache, the split file and (as settings_hash) the report. They are defined here, so
# re-running this cell updates every one of them; cells 4 and 7 only use these names.
TAR_PATH = Path(DRIVE_ROOT, "cache", f"rdd2022_yolo_{SETTINGS_HASH}.tar")
SPLIT_FILE = Path(DRIVE_ROOT, "splits", f"split_{SETTINGS_HASH}.json")
# The 13 GB download starts only if this is True and there is neither a matching cache nor raw data.
FORCE_DOWNLOAD = False

ULTRALYTICS_VERSION = "8.4.182"

# RDD2022 on figshare (article 21431547, DOI 10.6084/m9.figshare.21431547)
FIGSHARE_ARTICLE = 21431547
RDD_URL = "https://ndownloader.figshare.com/files/38030910"
RDD_MD5 = "b62bd51d2ffcfaa76c60f234f0cc2bb3"
RDD_SIZE = 13264172619
RAW_ZIP = "/content/rdd2022.zip"
RAW_DIR = "/content/rdd2022_raw"

# Drive and the model version
from google.colab import drive

drive.mount("/content/drive")
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
version_file = Path(DRIVE_ROOT, f"model_version_{RUN_MODE}.json")
if version_file.exists():
    MODEL_VERSION = json.loads(version_file.read_text())["model_version"]
    print("Model version read from", version_file.name)
else:
    MODEL_VERSION = "m-" + datetime.date.today().isoformat() + ("" if SMOKE else f"-{RUN_MODE}")
    tmp = version_file.with_suffix(".tmp")
    tmp.write_text(json.dumps({"model_version": MODEL_VERSION, "run_mode": RUN_MODE, "created": datetime.datetime.now(datetime.timezone.utc).isoformat()}))
    os.replace(tmp, version_file)
    print("Model version written to", version_file.name)

print(f"RUN_MODE={RUN_MODE} epochs={EPOCHS} fraction={FRACTION} version={MODEL_VERSION} settings_hash={SETTINGS_HASH}")

## 2. Environment
Stops if no GPU is attached. Uses Colab's own PyTorch and installs only `ultralytics`, pinned to one exact version.

In [ ]:
import platform, subprocess, sys

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU attached. In Colab: Runtime > Change runtime type > pick a GPU, then run again.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}"], check=True)

import numpy
import ultralytics

import importlib.metadata, sys


def loaded_vs_installed(packages):
    """{dist name: {"loaded": version of the module in this process or None, "installed": pip metadata version or None}}.
    Prints a warning for every package where both exist and differ (the process needs a restart then)."""
    result = {}
    for dist, module in packages.items():
        try:
            installed_v = importlib.metadata.version(dist)
        except importlib.metadata.PackageNotFoundError:
            installed_v = None
        mod = sys.modules.get(module)
        loaded_v = getattr(mod, "__version__", None) if mod else None
        result[dist] = {"loaded": loaded_v, "installed": installed_v}
        print(f"  {dist:15s} loaded {loaded_v or 'not loaded':12s} installed {installed_v or 'not installed'}")
        if loaded_v and installed_v and loaded_v.split("+")[0] != installed_v.split("+")[0]:
            print(f"  WARNING: {dist} loaded {loaded_v} but installed {installed_v}. The running process uses the loaded one; restart the runtime to use the installed one.")
    return result


print("python     ", platform.python_version())
print("CUDA       ", torch.version.cuda)
print("GPU        ", torch.cuda.get_device_name(0))
CORE_VERSIONS = loaded_vs_installed({"torch": "torch", "numpy": "numpy", "ultralytics": "ultralytics"})
if ultralytics.__version__ != ULTRALYTICS_VERSION:
    raise RuntimeError(f"Expected ultralytics {ULTRALYTICS_VERSION}, got {ultralytics.__version__}. Restart the runtime and run again.")

## 3. Mount Drive
Mounts Google Drive (a no-op if cell 1 already did) and creates the output folders under `DRIVE_ROOT`.

In [ ]:
drive.mount("/content/drive")
for sub in ("cache", "runs", "exports", "splits"):
    Path(DRIVE_ROOT, sub).mkdir(parents=True, exist_ok=True)
print("Drive folders ready under", DRIVE_ROOT)

## 4. Download and cache
`smoke` mode skips cells 4 to 9. Otherwise: if a converted dataset built with the same settings is cached on Drive (`cache/rdd2022_yolo_<hash>.tar`), copy it here and skip cells 5 to 7. Caches built with other settings are ignored. If there is none and there is no raw data either, the cell stops with a message unless `FORCE_DOWNLOAD = True` (cell 1). Otherwise it gets the raw data. The outer zip holds one inner zip per country (`RDD2022/<Country>.zip`). If the inner zips (or extracted country folders) are already under `RAW_DIR`, nothing is downloaded. Otherwise the outer zip is downloaded to `/content` (not Drive), its MD5 is checked and it is unzipped; the URL and MD5 are hard-coded, and if they fail the figshare API is asked at run time. Then each inner zip is extracted into `RAW_DIR/<Country>/...`, its file counts are checked against the zip listing, a marker `.extracted_<Country>` is written and the inner zip is deleted to free space. The final marker is `.inner_extracted_v2`.

In [ ]:
import collections, shutil, urllib.request

import pandas as pd
from PIL import Image

IMG_EXT = {".jpg", ".jpeg", ".png"}


def yolo_counts(split):
    """Images, background images and boxes per class for one split of LOCAL_DATA."""
    images = [p for p in Path(LOCAL_DATA, "images", split).iterdir() if p.suffix.lower() in IMG_EXT]
    boxes = [0] * len(CLASS_NAMES)
    background = 0
    for img in images:
        lines = Path(LOCAL_DATA, "labels", split, img.stem + ".txt").read_text().split("\n")
        lines = [ln for ln in lines if ln.strip()]
        background += not lines
        for ln in lines:
            boxes[int(ln.split()[0])] += 1
    return {"images": len(images), "background_images": background, "boxes": dict(zip(CLASS_NAMES, boxes))}


def md5_of(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""):
            h.update(chunk)
    return h.hexdigest()


def api_source():
    with urllib.request.urlopen(f"https://api.figshare.com/v2/articles/{FIGSHARE_ARTICLE}", timeout=60) as r:
        files = json.load(r)["files"]
    zips = [f for f in files if f["name"].lower().endswith(".zip")]
    if len(zips) != 1:
        raise RuntimeError(f"Expected one zip in the figshare article, found {[f['name'] for f in files]}")
    z = zips[0]
    return z["download_url"], z["computed_md5"], z["size"]


DATA_READY = False
if SMOKE:
    DATA_READY = True
    print("RUN_MODE is smoke: no dataset needed. Cells 4 to 9 are skipped.")
elif TAR_PATH.exists():
    others = sorted(p.name for p in TAR_PATH.parent.glob("rdd2022_yolo_*.tar") if p != TAR_PATH)
    if others:
        print("Ignoring caches built with other settings:", others)
    print("Found cached dataset on Drive, copying and extracting ...")
    shutil.rmtree(LOCAL_DATA, ignore_errors=True)
    shutil.copy(TAR_PATH, "/content/rdd2022_yolo.tar")
    subprocess.run(["tar", "-xf", "/content/rdd2022_yolo.tar", "-C", str(Path(LOCAL_DATA).parent)], check=True)
    os.remove("/content/rdd2022_yolo.tar")
    meta = Path(LOCAL_DATA, "build_settings.json")
    if not Path(LOCAL_DATA, "data.yaml").exists() or not meta.exists() or json.loads(meta.read_text()) != BUILD_SETTINGS:
        raise RuntimeError(f"{TAR_PATH.name} does not match the current settings or is incomplete. Delete it on Drive and run again.")
    DATA_READY = True
    print("Dataset ready at", LOCAL_DATA)
else:
    import zipfile

    EXPECTED_COUNTRIES = {"China_Drone", "China_MotorBike", "Czech", "India", "Japan", "Norway", "United_States"}
    raw = Path(RAW_DIR)
    marker = raw / ".inner_extracted_v2"
    if marker.exists():
        print("All inner zips were already extracted into", RAW_DIR)
    else:
        inner_zips = sorted(raw.rglob("*.zip")) if raw.exists() else []  # RAW_DIR/RDD2022/<Country>.zip
        if not inner_zips and not any(raw.glob(".extracted_*")):
            if not FORCE_DOWNLOAD:
                cached = sorted(p.name for p in TAR_PATH.parent.glob("*.tar"))
                raise RuntimeError(
                    f"No cache named {TAR_PATH.name} and no raw data under {RAW_DIR}. Not downloading 13 GB. "
                    f"Cache files on Drive: {cached or 'none'}. If one was built with the same settings, rename it to {TAR_PATH.name} "
                    f"(its build_settings.json is checked on load). Otherwise set FORCE_DOWNLOAD = True in cell 1 and run again."
                )
            free_gb = shutil.disk_usage("/content").free / 1e9
            if free_gb < 40:
                raise RuntimeError(f"Only {free_gb:.0f} GB free on /content; need about 40 GB for the zip and its contents.")
            sources = [(RDD_URL, RDD_MD5, RDD_SIZE)]
            good = False
            for attempt in range(2):
                url, md5, size = sources[-1]
                print("Downloading", url)
                rc = subprocess.run(["wget", "-c", "-q", "--show-progress", "-O", RAW_ZIP, url]).returncode
                if rc == 0 and Path(RAW_ZIP).exists() and md5_of(RAW_ZIP) == md5:
                    good = True
                    break
                print(f"Download or MD5 check failed (wget exit {rc}). Asking the figshare API.")
                Path(RAW_ZIP).unlink(missing_ok=True)
                if attempt == 0:
                    sources.append(api_source())
            if not good:
                raise RuntimeError("Could not download RDD2022 with a matching MD5. Check the figshare page and the config cell.")
            print("MD5 OK, unzipping the outer zip ...")
            shutil.rmtree(RAW_DIR, ignore_errors=True)
            subprocess.run(["unzip", "-q", "-o", RAW_ZIP, "-d", RAW_DIR], check=True)
            inner_zips = sorted(raw.rglob("*.zip"))

        remaining = [z for z in inner_zips if not (raw / f".extracted_{z.stem}").exists()]
        need_gb = sum(z.stat().st_size for z in remaining) / 1e9 + 5
        free_gb = shutil.disk_usage("/content").free / 1e9
        print(f"Free disk: {free_gb:.1f} GB. Remaining inner zips: {[z.stem for z in remaining]}, need {need_gb:.1f} GB (their size + 5 GB).")
        if free_gb < need_gb:
            raise RuntimeError(f"Only {free_gb:.1f} GB free but {need_gb:.1f} GB needed to extract the remaining inner zips. Free some space (Runtime > Manage sessions, or delete files under /content) and run again.")

        for n, z in enumerate(remaining, 1):
            country = z.stem
            print(f"[{n}/{len(remaining)}] extracting {country} ({z.stat().st_size / 1e9:.2f} GB) ...")
            with zipfile.ZipFile(z) as zf:
                files = [i.filename for i in zf.infolist() if not i.is_dir()]
                stray = [f for f in files if not f.startswith(country + "/")]
                if stray:
                    raise RuntimeError(f"{z.name} has entries outside {country}/, for example {stray[:3]}.")
                want = {ext: sum(f.lower().endswith(ext) for f in files) for ext in (".jpg", ".xml")}
                zf.extractall(raw)
            got = {ext: sum(p.suffix.lower() == ext for p in (raw / country).rglob("*") if p.is_file()) for ext in (".jpg", ".xml")}
            if got != want:
                raise RuntimeError(f"{country}: files on disk {got} do not match the zip listing {want}. Delete {raw / country} and run again.")
            (raw / f".extracted_{country}").touch()
            z.unlink()
            print(f"    {country}: {got['.jpg']} jpg, {got['.xml']} xml OK; zip deleted; free disk {shutil.disk_usage('/content').free / 1e9:.1f} GB")

        done = {p.name[len(".extracted_"):] for p in raw.glob(".extracted_*")}
        if EXPECTED_COUNTRIES - done:
            raise RuntimeError(f"Countries not extracted: {sorted(EXPECTED_COUNTRIES - done)}. Found: {sorted(done)}.")
        marker.touch()
        Path(RAW_ZIP).unlink(missing_ok=True)
        print(f"Outer zip deleted if present. Free disk: {shutil.disk_usage('/content').free / 1e9:.1f} GB")
    print("Raw dataset at", RAW_DIR)

## 5. Inventory
Layout, as confirmed from the zip listings on 19 Oct: `RAW_DIR/<Country>/train/images/*.jpg`, `RAW_DIR/<Country>/train/annotations/xmls/*.xml` (one per train image) and `RAW_DIR/<Country>/test/images/*.jpg` (the test split has no annotations). Images and XMLs are matched by file stem. Prints, per country, the image and XML counts, the unmatched files and the test counts, then a class count table. Lists every country and marks the excluded ones. Stops on any class name (in the included countries) that is not in the mapping or in `DROP_CODES`.

In [ ]:
if DATA_READY:
    print('Dataset cells are skipped (smoke mode or cached dataset).')
else:
    import xml.etree.ElementTree as ET

    raw = Path(RAW_DIR)
    found = sorted(str(p.relative_to(raw)) for p in raw.glob("*/*/*") if p.is_dir())  # depth 3
    parts = {}  # (country, split) -> images and xml files
    for cdir in sorted(p for p in raw.iterdir() if p.is_dir()):
        for split in ("train", "test"):
            img_dir = cdir / split / "images"
            if img_dir.is_dir():
                parts[(cdir.name, split)] = {
                    "images": sorted(p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXT),
                    "xmls": sorted((cdir / split).rglob("*.xml")),
                }
    all_countries = sorted({c for c, _ in parts})
    if not any(s == "train" for _, s in parts):
        raise RuntimeError(f"No <country>/train/images folders found under {RAW_DIR}. Folders found at depth 3: {found}")
    not_found = [c for c in EXCLUDE_COUNTRIES if c not in all_countries]
    if not_found:
        print(f"WARNING: EXCLUDE_COUNTRIES {not_found} match no folder. Real folder names: {all_countries}")
    ACTIVE_COUNTRIES = [c for c in all_countries if c not in EXCLUDE_COUNTRIES and (c, "train") in parts]
    if not ACTIVE_COUNTRIES:
        raise RuntimeError("No country is left after EXCLUDE_COUNTRIES.")

    layout = []
    for country in all_countries:
        tr, te = parts.get((country, "train")), parts.get((country, "test"))
        tr_img, tr_xml = {p.stem for p in tr["images"]}, {p.stem for p in tr["xmls"]}
        layout.append({
            "country": country, "excluded": country in EXCLUDE_COUNTRIES,
            "train_images": len(tr["images"]), "train_xmls": len(tr["xmls"]),
            "images_without_xml": len(tr_img - tr_xml), "xmls_without_image": len(tr_xml - tr_img),
            "test_images": len(te["images"]) if te else 0, "test_xmls": len(te["xmls"]) if te else 0,
        })
        if tr_img != tr_xml:
            print(f"WARNING: {country}: {len(tr_img - tr_xml)} train images without an XML, {len(tr_xml - tr_img)} XMLs without an image (both expected 0)")
    print(pd.DataFrame(layout).to_string(index=False))

    TEST_HAS_LABELS = any(r["test_xmls"] > 0 for r in layout)
    print("\ntest/ folders contain annotations:", TEST_HAS_LABELS, "(expected False; test images are never used)")

    rows, names_seen, names_excluded = [], set(), set()
    for (country, split), part in sorted(parts.items()):
        if not part["xmls"]:
            continue
        counts = collections.Counter()
        for x in part["xmls"]:
            for obj in ET.parse(x).getroot().iter("object"):
                counts[obj.findtext("name").strip()] += 1
        (names_excluded if country in EXCLUDE_COUNTRIES else names_seen).update(counts)
        rows.append({"country": country, "split": split, "excluded": country in EXCLUDE_COUNTRIES, **counts})
    table = pd.DataFrame(rows).fillna(0)
    for col in table.columns[3:]:
        table[col] = table[col].astype(int)
    print(table.to_string(index=False))

    unknown = names_seen - set(CODE_TO_CLASS) - DROP_CODES
    if unknown:
        raise RuntimeError(f"Unknown class names in the XML files: {sorted(unknown)}. Add each to CODE_TO_CLASS or DROP_CODES (cell 1), then run again.")
    print("Unmapped names seen only in excluded countries (ignored):", sorted(names_excluded - set(CODE_TO_CLASS) - DROP_CODES))
    for country in ACTIVE_COUNTRIES:
        if not parts[(country, "train")]["xmls"]:
            raise RuntimeError(f"{country}/train has no XML annotations. Folders found at depth 3: {found}")
    print("All class names in the included countries are mapped or dropped. Included:", ACTIVE_COUNTRIES)

## 6. Convert Pascal VOC XML to YOLO txt
Skips excluded countries. Reads the real image size from each image, clips boxes to it and drops boxes with no area. Images with no kept boxes stay as background images with an empty label file. Every image is downscaled so its longest side is at most 640 px (never upscaled), JPEG quality 90. Files are written under a temporary name and renamed when complete, and images whose output already exists are skipped, so a re-run continues where it stopped. Only labelled train images are used; `test/` is ignored.

In [ ]:
if DATA_READY:
    print('Dataset cells are skipped (smoke mode or cached dataset).')
else:
    import multiprocessing as mp
    import time

    ALL_DIR = Path(LOCAL_DATA, "_all")
    MAX_SIDE = IMGSZ


    def convert_one(job):
        country, img_path, xml_path, out_img, out_lbl = job
        img = Image.open(img_path)
        w, h = img.size  # real size, not the XML's; opening is lazy so this is cheap
        stats = collections.Counter()
        lines = []
        if xml_path is None:
            stats["no_xml"] += 1
        else:
            for obj in ET.parse(xml_path).getroot().iter("object"):
                code = obj.findtext("name").strip()
                if code in DROP_CODES:
                    stats["dropped_class"] += 1
                    continue
                bb = obj.find("bndbox")
                x1, y1, x2, y2 = (float(bb.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax"))
                x1, x2 = min(max(x1, 0.0), w), min(max(x2, 0.0), w)
                y1, y2 = min(max(y1, 0.0), h), min(max(y2, 0.0), h)
                if x2 <= x1 or y2 <= y1:
                    stats["dropped_area"] += 1
                    continue
                cls = CODE_TO_CLASS[code]
                lines.append(f"{cls} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}")
                stats[f"box_{CLASS_NAMES[cls]}"] += 1
        if not lines:
            stats["background"] += 1
        if os.path.exists(out_img) and os.path.exists(out_lbl):
            stats["reused"] += 1  # the image is renamed last, so its existence means both files are complete
            return country, Path(img_path).stem, stats
        scale = MAX_SIDE / max(w, h)
        if scale < 1:
            nw, nh = max(1, round(w * scale)), max(1, round(h * scale))
            img.draft("RGB", (nw, nh))  # fast JPEG decode at reduced size
            img = img.convert("RGB").resize((nw, nh), Image.BILINEAR)
        else:
            img = img.convert("RGB")
        tmp_lbl, tmp_img = out_lbl + ".tmp", out_img + ".tmp"
        Path(tmp_lbl).write_text("\n".join(lines) + ("\n" if lines else ""))
        os.replace(tmp_lbl, out_lbl)
        img.save(tmp_img, "JPEG", quality=90)
        os.replace(tmp_img, out_img)
        return country, Path(img_path).stem, stats


    settings_file = ALL_DIR / "build_settings.json"
    if ALL_DIR.exists() and (not settings_file.exists() or json.loads(settings_file.read_text()) != BUILD_SETTINGS):
        print("Partial conversion was made with other settings; starting over.")
        shutil.rmtree(LOCAL_DATA, ignore_errors=True)
    (ALL_DIR / "images").mkdir(parents=True, exist_ok=True)
    (ALL_DIR / "labels").mkdir(parents=True, exist_ok=True)
    settings_file.write_text(json.dumps(BUILD_SETTINGS))
    for tmp in ALL_DIR.rglob("*.tmp"):
        tmp.unlink()  # half-written files from a disconnect

    jobs, stems = [], collections.Counter()
    for country in ACTIVE_COUNTRIES:
        part = parts[(country, "train")]
        xml_by_stem = {x.stem: x for x in part["xmls"]}
        orphans = set(xml_by_stem) - {p.stem for p in part["images"]}
        if orphans:
            print(f"{country}: {len(orphans)} XML files have no image and are ignored")
        for p in part["images"]:
            stems[p.stem] += 1
            jobs.append((country, str(p), str(xml_by_stem[p.stem]) if p.stem in xml_by_stem else None,
                         str(ALL_DIR / "images" / (p.stem + ".jpg")), str(ALL_DIR / "labels" / (p.stem + ".txt"))))
    assert all("/train/" in j[1] for j in jobs), "Only train images are converted; test images have no labels."
    dupes = [s for s, n in stems.items() if n > 1]
    assert not dupes, f"Image filenames are not unique across countries, for example {dupes[:5]}"

    per_country = collections.defaultdict(collections.Counter)
    manifest = {job[1].rsplit("/", 1)[-1].rsplit(".", 1)[0]: job[0] for job in jobs}
    done, t0 = 0, time.time()
    with mp.Pool(os.cpu_count() or 2) as pool:
        for country, stem, stats in pool.imap_unordered(convert_one, jobs, chunksize=16):
            per_country[country].update(stats)
            per_country[country]["images"] += 1
            done += 1
            if done % 2000 == 0 or done == len(jobs):
                print(f"{done}/{len(jobs)} images, {time.time() - t0:.0f} s")

    summary = pd.DataFrame(per_country).T.fillna(0).astype(int)
    print(summary.to_string())
    print("Dropped boxes: class not used =", int(summary.get("dropped_class", pd.Series(dtype=int)).sum()),
          ", zero area =", int(summary.get("dropped_area", pd.Series(dtype=int)).sum()))
    no_xml = int(summary.get("no_xml", pd.Series(dtype=int)).sum())
    if no_xml > 0.05 * len(jobs):
        raise RuntimeError(f"{no_xml} of {len(jobs)} train images have no XML file. That is too many to treat as background; check the layout.")
    (ALL_DIR / "manifest.json").write_text(json.dumps(manifest))
    print(f"Converted {len(jobs)} images.")

## 7. Split and cache
Splits train into train and val per country with `VAL_FRACTION`, as a block split: the images of each country are sorted by file name and cut into blocks of `SPLIT_BLOCK` (100) consecutive images, and whole blocks are assigned to val with the seed until val is about 10% of the country. Reason: consecutive frames can look alike, so a random per-image split could put near duplicates in both train and val and inflate mAP. The split lists are saved to Drive (named with the settings hash) and reused if they exist, so val never changes between sessions. Writes `data.yaml`, tars the dataset to Drive `cache/`, prints counts and deletes the raw zip.

In [ ]:
if DATA_READY:
    print('Dataset cells are skipped (smoke mode or cached dataset).')
else:
    import random

    split_file = SPLIT_FILE
    manifest = json.loads((ALL_DIR / "manifest.json").read_text())

    if split_file.exists():
        split = json.loads(split_file.read_text())
        print("Reusing saved split", split_file.name)
    else:
        split = {"train": [], "val": []}
        print(f"Block split: blocks of {SPLIT_BLOCK} consecutive images by file name. It is NOT known whether file numbers follow capture order, so a block may or may not be one stretch of road.")
        for country in sorted(set(manifest.values())):
            stems = sorted(s for s, c in manifest.items() if c == country)
            blocks = [stems[i:i + SPLIT_BLOCK] for i in range(0, len(stems), SPLIT_BLOCK)]
            random.Random(f"{SEED}-{country}").shuffle(blocks)
            target, val_stems = round(len(stems) * VAL_FRACTION), []
            for b in blocks:
                if abs(target - (len(val_stems) + len(b))) < abs(target - len(val_stems)):  # take the block only if it gets closer
                    val_stems += b
            val_set = set(val_stems)
            split["val"] += val_stems
            split["train"] += [s for s in stems if s not in val_set]
            print(f"  {country}: {len(stems)} images in {len(blocks)} blocks -> val {len(val_stems)} ({len(val_stems) / len(stems):.1%}), train {len(stems) - len(val_stems)}")
        split["train"].sort()
        split["val"].sort()
        split_file.write_text(json.dumps(split))
        print("Saved new split", split_file.name)
    if set(split["train"]) | set(split["val"]) != set(manifest) or set(split["train"]) & set(split["val"]):
        raise RuntimeError(f"{split_file} does not match the converted images. Delete it on Drive only if you accept a new val set, then run again.")

    for name in ("train", "val"):
        for kind in ("images", "labels"):
            Path(LOCAL_DATA, kind, name).mkdir(parents=True, exist_ok=True)
        for stem in split[name]:
            for kind, ext in (("images", ".jpg"), ("labels", ".txt")):
                src, dst = ALL_DIR / kind / f"{stem}{ext}", Path(LOCAL_DATA, kind, name, f"{stem}{ext}")
                if src.exists():  # a re-run after a disconnect finds some files already moved
                    os.replace(src, dst)
                elif not dst.exists():
                    raise RuntimeError(f"Missing {dst.name}. Delete {LOCAL_DATA} and run cells 4 to 7 again.")
    shutil.rmtree(ALL_DIR)

    Path(LOCAL_DATA, "data.yaml").write_text(
        f"path: {LOCAL_DATA}\ntrain: images/train\nval: images/val\nnames:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASS_NAMES))
    )
    Path(LOCAL_DATA, "build_settings.json").write_text(json.dumps(BUILD_SETTINGS))

    tmp_tar = Path(str(TAR_PATH) + ".part")
    subprocess.run(["tar", "-cf", str(tmp_tar), "-C", str(Path(LOCAL_DATA).parent), Path(LOCAL_DATA).name], check=True)
    os.replace(tmp_tar, TAR_PATH)
    print("Cached to", TAR_PATH, f"({TAR_PATH.stat().st_size / 1e9:.2f} GB)")

    for name in ("train", "val"):
        print(name, yolo_counts(name))

    Path(RAW_ZIP).unlink(missing_ok=True)
    shutil.rmtree(RAW_DIR, ignore_errors=True)
    DATA_READY = True

## 8. Train
In `smoke` mode this cell does nothing. Otherwise it writes to `DRIVE_ROOT/runs/<RUN_MODE>`, so `last.pt` lands on Drive every epoch. It prints which of three cases applies. Ultralytics 8.4.182 (`engine/trainer.py`, `resume_training`) can resume only if `0 < ckpt["epoch"] + 1 < epochs`; at the end of training `strip_optimizer` sets `epoch = -1` in `last.pt`. So a `last.pt` with `epoch == -1`, or whose last epoch was the final one, belongs to a finished run.

In [ ]:
import time

from ultralytics import YOLO

RUN_DIR = Path(DRIVE_ROOT, "runs", RUN_MODE)
LAST, BEST = RUN_DIR / "weights" / "last.pt", RUN_DIR / "weights" / "best.pt"
DATA_YAML = str(Path(LOCAL_DATA, "data.yaml"))


def epochs_done():
    csv = RUN_DIR / "results.csv"
    return max(0, len(csv.read_text().strip().split("\n")) - 1) if csv.exists() else 0


def run_state():
    """'fresh' (no last.pt), 'finished' (cannot be resumed) or 'unfinished' (resumable), by the trainer's own rule."""
    if not LAST.exists():
        return "fresh"
    ckpt = torch.load(LAST, map_location="cpu", weights_only=False)
    epoch = ckpt.get("epoch", -1)
    total = (ckpt.get("train_args") or {}).get("epochs", EPOCHS)
    print(f"last.pt: epoch field = {epoch}, planned epochs = {total}")
    return "finished" if not (0 < epoch + 1 < total) else "unfinished"


_t = {}
def _start(trainer):
    _t["t0"] = time.time()
def _end(trainer):
    print(f"epoch {trainer.epoch + 1}/{trainer.epochs} took {time.time() - _t['t0']:.0f} s")


if SMOKE:
    print("RUN_MODE is smoke: no training.")
else:
    if not Path(DATA_YAML).exists():
        raise RuntimeError("data.yaml is missing. Run cells 4 to 7 first.")
    state = run_state()
    if state == "finished":
        print(f"CASE: finished run. Skipping training and using {BEST}.")
    elif state == "unfinished":
        print(f"CASE: unfinished run. Resuming from {LAST} after {epochs_done()} epochs.")
        model = YOLO(str(LAST))
        model.add_callback("on_train_epoch_start", _start)
        model.add_callback("on_train_epoch_end", _end)
        model.train(resume=True)
    else:
        print("CASE: no last.pt. Starting a fresh run.")
        model = YOLO(MODEL)
        model.add_callback("on_train_epoch_start", _start)
        model.add_callback("on_train_epoch_end", _end)
        model.train(
            data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, seed=SEED, batch=-1, cache=False,
            fraction=FRACTION, project=str(Path(DRIVE_ROOT, "runs")), name=RUN_MODE, exist_ok=True, workers=2,
        )
    if not BEST.exists():
        raise RuntimeError(f"{BEST} was not written. Training did not finish; run this cell again to resume.")

## 9. Validate
In `smoke` mode this cell does nothing. Otherwise it validates `best.pt` on the held-out val part and prints mAP per class, overall mAP and the val counts.

In [ ]:
if SMOKE:
    print("RUN_MODE is smoke: no validation (the weights are random).")
    val_summary, val_counts, train_counts = None, None, None
else:
    model = YOLO(str(BEST))
    metrics = model.val(data=DATA_YAML, imgsz=IMGSZ, split="val", plots=False, project=str(Path(DRIVE_ROOT, "runs")), name=f"{RUN_MODE}_val", exist_ok=True)
    val_counts = yolo_counts("val")
    train_counts = yolo_counts("train")
    ap50 = {CLASS_NAMES[int(c)]: float(metrics.box.ap50[i]) for i, c in enumerate(metrics.box.ap_class_index)}
    val_summary = {
        "map50_per_class": ap50,
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map),
    }
    table = pd.DataFrame(
        [{"class": n, "mAP@0.5": ap50.get(n), "val boxes": val_counts["boxes"][n]} for n in CLASS_NAMES]
        + [{"class": "ALL", "mAP@0.5": val_summary["map50"], "val boxes": sum(val_counts["boxes"].values())}]
    )
    print(table.to_string(index=False))
    print(f"mAP@0.5:0.95 overall: {val_summary['map50_95']:.4f}")
    print(f"val images: {val_counts['images']} ({val_counts['background_images']} background)")

## 10. Export
Exports to float16 TFLite with no NMS, in a local work folder, then copies the file to Drive `exports/`. In Ultralytics 8.4.182, `format="tflite"` is deprecated and silently becomes `litert`, which writes a float32 model, and `half=True` is a deprecated alias of `quantize=16`, which the TensorFlow formats reject. So this cell uses `format="saved_model"` (ONNX, then onnx2tf), which writes both a float32 and a float16 `.tflite`, and keeps the float16 one. In `smoke` mode the weights are random (`birdeye_smoke_random_float16.tflite`). Prints, for each key package, the version loaded in the running process and the installed version, with a warning if they differ, and lists every package the export installed or changed.

In [ ]:
import importlib, importlib.metadata


def installed():
    return {d.metadata["Name"].lower(): d.version for d in importlib.metadata.distributions() if d.metadata["Name"]}


WORK = Path("/content/export_work")
if SMOKE:
    TFLITE_PATH = Path(DRIVE_ROOT, "exports", "birdeye_smoke_random_float16.tflite")
    TORCH_WEIGHTS = WORK / "best.pt"
    WEIGHTS_NOTE = "random, not for detection"
else:
    TFLITE_PATH = Path(DRIVE_ROOT, "exports", f"birdeye_{MODEL_VERSION}_{RUN_MODE}_float16.tflite")
    TORCH_WEIGHTS = BEST
    WEIGHTS_NOTE = f"trained: {BEST}"
before = installed()

if TFLITE_PATH.exists() and not SMOKE:
    print("Already exported:", TFLITE_PATH)
else:
    shutil.rmtree(WORK, ignore_errors=True)
    WORK.mkdir()
    if SMOKE:
        from ultralytics.nn.tasks import DetectionModel

        torch.manual_seed(SEED)
        net = DetectionModel("yolo11n.yaml", nc=len(CLASS_NAMES), verbose=False)  # random weights
        net.names = dict(enumerate(CLASS_NAMES))
        torch.save({"model": net.half(), "train_args": {"imgsz": IMGSZ, "task": "detect", "epochs": 0}, "epoch": -1}, TORCH_WEIGHTS)
    else:
        shutil.copy(BEST, WORK / "best.pt")
    YOLO(str(WORK / "best.pt")).export(format="saved_model", imgsz=IMGSZ)
    f16, f32 = sorted(WORK.rglob("*_float16.tflite")), sorted(WORK.rglob("*_float32.tflite"))
    if not f16:
        raise RuntimeError(f"Export finished but no *_float16.tflite found under {WORK}: {sorted(str(p) for p in WORK.rglob('*.tflite'))}")
    if f32 and f16[0].stat().st_size > 0.7 * f32[0].stat().st_size:
        raise RuntimeError(f"The float16 file ({f16[0].stat().st_size / 1e6:.1f} MB) is not much smaller than the float32 file ({f32[0].stat().st_size / 1e6:.1f} MB); it may not be float16.")
    shutil.copy(f16[0], TFLITE_PATH)
    print("Wrote", TFLITE_PATH, "| float16", f"{f16[0].stat().st_size / 1e6:.2f} MB", "| float32", f"{f32[0].stat().st_size / 1e6:.2f} MB" if f32 else "n/a")

importlib.invalidate_caches()
after = installed()
print("Key packages (loaded = module in this process, installed = pip metadata):")
PACKAGE_VERSIONS = loaded_vs_installed({
    "torch": "torch", "numpy": "numpy", "ultralytics": "ultralytics", "tensorflow": "tensorflow", "onnx": "onnx",
    "onnx2tf": "onnx2tf", "onnxslim": "onnxslim", "onnxruntime": "onnxruntime", "ai-edge-litert": "ai_edge_litert",
    "litert-torch": "litert_torch",
})
PACKAGE_VERSIONS["python"] = platform.python_version()
changed = {k: v for k, v in after.items() if before.get(k) != v}
PACKAGE_VERSIONS["changed_by_export"] = changed
if changed:
    print("Installed or changed by the export:", changed)
else:
    print("Nothing new was installed in this session (export may have been done earlier).")

## 11. Inspect the TFLite and decide its conventions from data
Ultralytics clips its boxes to the image frame, so the TFLite box is clipped to [0, 640] before the IoU, the coordinate difference and the agree flag. Runs 20 val images through the TFLite file and through the PyTorch weights with the same preprocessing, and decides from the numbers: input layout, channel order, pixel range, output layout, box format, whether boxes are normalised, and whether scores are probabilities or logits. Then prints the top detection from both side by side for 5 images and saves 3 overlay images to Drive `exports/`. In `smoke` mode the weights are random, so the detections are noise; the numeric agreement between TFLite and PyTorch is still meaningful. The test images are letterboxed with Ultralytics' own rule (centred, pad value 114, `cv2.INTER_LINEAR`), read from its 8.4.182 source. The cell also maps the TFLite box back to the original image and compares it with PyTorch's own mapping. Everything decided here goes into the report.

In [ ]:
import random

import cv2
import numpy as np
from PIL import ImageDraw

try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    import tensorflow as tf
    Interpreter = tf.lite.Interpreter

NC, AGREE_PX, N_IMAGES = len(CLASS_NAMES), 5.0, 20
interp = Interpreter(model_path=str(TFLITE_PATH))
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
in_shape, out_shape = inp["shape"].tolist(), out["shape"].tolist()
TFLITE_SIZE_MB = TFLITE_PATH.stat().st_size / 1e6
print("input ", in_shape, np.dtype(inp["dtype"]).name)
print("output", out_shape, np.dtype(out["dtype"]).name)
print(f"size   {TFLITE_SIZE_MB:.2f} MB")
if np.dtype(inp["dtype"]) != np.float32:
    raise RuntimeError(f"Input dtype is {np.dtype(inp['dtype']).name}, not float32. This check feeds float32; adjust cell 11.")
if in_shape[1:] == [IMGSZ, IMGSZ, 3]:
    IN_LAYOUT = "NHWC"
elif in_shape[1:] == [3, IMGSZ, IMGSZ]:
    IN_LAYOUT = "NCHW"
else:
    raise RuntimeError(f"Unexpected input shape {in_shape} for imgsz {IMGSZ}.")
if out_shape[1] == 4 + NC and out_shape[2] != 4 + NC:
    CHANNELS_FIRST = True
elif out_shape[2] == 4 + NC and out_shape[1] != 4 + NC:
    CHANNELS_FIRST = False
else:
    raise RuntimeError(f"Cannot tell the layout of output shape {out_shape} (expected 4 + {NC} = {4 + NC} on one axis).")
OUT_LAYOUT = "channels_first [1, 4+classes, anchors]" if CHANNELS_FIRST else "channels_last [1, anchors, 4+classes]"
anchors_expected = sum((IMGSZ // s) ** 2 for s in (8, 16, 32))
n_anchors = out_shape[2] if CHANNELS_FIRST else out_shape[1]
print("output layout:", OUT_LAYOUT, "| anchors", n_anchors, "(expected", anchors_expected, ")")
if n_anchors != anchors_expected:
    print("WARNING: unexpected anchor count; the model may include NMS or a different head.")


def to_anchors(raw):
    return raw.T if CHANNELS_FIRST else raw  # (anchors, 4 + classes)


def letterbox(img):
    """Ultralytics LetterBox rule (8.4.182, data/augment.py), as used by predict: scale = IMGSZ / max(w, h), cv2.INTER_LINEAR,
    pad value 114, centred with left/top = round((IMGSZ - new_size) / 2 - 0.1). Returns the canvas and the numbers needed to undo it."""
    a = np.asarray(img)  # RGB
    h, w = a.shape[:2]
    scale = IMGSZ / max(w, h)
    new_w, new_h = round(w * scale), round(h * scale)
    if (new_w, new_h) != (w, h):
        a = cv2.resize(a, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    pad_x, pad_y = round((IMGSZ - new_w) / 2 - 0.1), round((IMGSZ - new_h) / 2 - 0.1)
    canvas = np.full((IMGSZ, IMGSZ, 3), 114, dtype=np.uint8)
    canvas[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = a
    return Image.fromarray(canvas), {"w": w, "h": h, "scale": scale, "new_w": new_w, "new_h": new_h, "pad_x": pad_x, "pad_y": pad_y}


def to_original(box, m):
    """Model pixels -> original image pixels, then clip to the original frame (Ultralytics scale_boxes does it in this order)."""
    gx, gy = m["new_w"] / m["w"], m["new_h"] / m["h"]
    x1, y1, x2, y2 = (box[0] - m["pad_x"]) / gx, (box[1] - m["pad_y"]) / gy, (box[2] - m["pad_x"]) / gx, (box[3] - m["pad_y"]) / gy
    return [min(max(x1, 0.0), m["w"]), min(max(y1, 0.0), m["h"]), min(max(x2, 0.0), m["w"]), min(max(y2, 0.0), m["h"])]


# 20 test images: (name, canvas, ground-truth boxes in canvas pixels, letterbox numbers, original image)
test = []
if SMOKE:
    assets = sorted((Path(ultralytics.__file__).parent / "assets").glob("*.jpg"))
    rng = np.random.default_rng(SEED)
    sources = [(p.name, Image.open(p).convert("RGB")) for p in assets]
    while len(sources) < N_IMAGES:
        sources.append((f"noise_{len(sources)}", Image.fromarray(rng.integers(0, 256, (IMGSZ, IMGSZ, 3), dtype=np.uint8))))
    for name, im in sources[:N_IMAGES]:
        canvas, meta = letterbox(im)
        test.append((name, canvas, [], meta, im))
else:
    lbls = [p for p in sorted(Path(LOCAL_DATA, "labels", "val").glob("*.txt")) if p.read_text().strip()]
    for lbl in random.Random(SEED).sample(lbls, min(N_IMAGES, len(lbls))):
        im = Image.open(Path(LOCAL_DATA, "images", "val", lbl.stem + ".jpg")).convert("RGB")
        canvas, meta = letterbox(im)
        gt = []
        for ln in lbl.read_text().strip().split("\n"):
            c, cx, cy, bw, bh = ln.split()
            cx, cy = float(cx) * meta["new_w"] + meta["pad_x"], float(cy) * meta["new_h"] + meta["pad_y"]
            bw, bh = float(bw) * meta["new_w"], float(bh) * meta["new_h"]
            gt.append((int(c), [cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2]))
        test.append((lbl.stem, canvas, gt, meta, im))
print(f"{len(test)} test images")

# PyTorch reference: the 640 x 640 canvas is already letterboxed with Ultralytics' own rule, so predict() adds no padding.
# A numpy array is treated as BGR by Ultralytics, so pass the canvas channel-reversed to give it RGB.
torch_model = YOLO(str(TORCH_WEIGHTS))


def torch_top(arr_rgb):
    r = torch_model.predict(arr_rgb[..., ::-1].copy(), imgsz=IMGSZ, conf=1e-4, iou=0.7, verbose=False, device="cpu")[0]
    if not len(r.boxes):
        return None
    k = int(r.boxes.conf.argmax())
    return {"cls": int(r.boxes.cls[k]), "conf": float(r.boxes.conf[k]), "xyxy": r.boxes.xyxy[k].tolist()}


ref = [torch_top(np.asarray(t[1])) for t in test]  # boxes in canvas pixels
ref_orig = [torch_top(np.asarray(t[4])) for t in test]  # PyTorch's own letterbox and mapping back to the original image
n_ref = sum(r is not None for r in ref)
print(f"PyTorch found a top detection in {n_ref} of {len(test)} images")
if n_ref < 5:
    raise RuntimeError("PyTorch found almost no detections, so the TFLite conventions cannot be decided from data.")
if SMOKE:
    print("NOTE: smoke mode. The weights are random, so every detection below is noise. Only the TFLite vs PyTorch agreement matters.")


def run_tflite(canvas, bgr, range255):
    a = np.asarray(canvas)
    a = (a[..., ::-1] if bgr else a).astype(np.float32)
    if not range255:
        a = a / 255.0
    if IN_LAYOUT == "NCHW":
        a = a.transpose(2, 0, 1)
    interp.set_tensor(inp["index"], a[None])
    interp.invoke()
    return interp.get_tensor(out["index"])[0].copy()


# 1) Input: which channel order and pixel range makes the TFLite top score match PyTorch's top score?
HYPOTHESES = [("RGB", "0-1"), ("BGR", "0-1"), ("RGB", "0-255"), ("BGR", "0-255")]
raws, score_gap = {}, {}
for ch, rng_ in HYPOTHESES:
    raws[(ch, rng_)] = [run_tflite(t[1], ch == "BGR", rng_ == "0-255") for t in test]
    gaps = []
    for raw, rf in zip(raws[(ch, rng_)], ref):
        if rf is None:
            continue
        s = to_anchors(raw)[:, 4:]
        top = float(s.max())
        gaps.append(abs((top if 0 <= s.min() and s.max() <= 1.001 else 1 / (1 + np.exp(-top))) - rf["conf"]))
    score_gap[(ch, rng_)] = float(np.mean(gaps))
    print(f"  input {ch} {rng_:6s}: mean |TFLite top score - PyTorch top score| = {score_gap[(ch, rng_)]:.5f}")
ranked = sorted(score_gap, key=score_gap.get)
best_h = ranked[0]
input_decided = score_gap[best_h] < 0.02 and score_gap[ranked[1]] > max(3 * score_gap[best_h], 0.02)
if not input_decided:
    print("WARNING: channel order and pixel range could not be told apart from the data. Assuming RGB 0-1; check by hand.")
    best_h = ("RGB", "0-1")
CHANNEL_ORDER, PIXEL_RANGE = best_h
print("DECIDED input:", IN_LAYOUT, CHANNEL_ORDER, PIXEL_RANGE, "" if input_decided else "(NOT decided)")

anch = [to_anchors(r) for r in raws[best_h]]

# 2) Scores: probabilities or logits
s_all = np.concatenate([a[:, 4:].ravel() for a in anch])
s_min, s_max = float(s_all.min()), float(s_all.max())
SCORES = "probabilities (0 to 1)" if s_min >= -1e-3 and s_max <= 1.001 else "raw logits"
print(f"DECIDED scores: {SCORES} (min {s_min:.4f}, max {s_max:.4f})")

# 3) Boxes: normalised or pixels, from the maximum box value over all images
box_max = float(max(a[:, :4].max() for a in anch))
NORMALISED = box_max <= 2.0
box_scale = IMGSZ if NORMALISED else 1
print(f"DECIDED box values: {'normalised (0 to 1)' if NORMALISED else f'pixels of the {IMGSZ} input'} (max box value over {len(anch)} images = {box_max:.2f})")


def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def to_xyxy(b, fmt):
    """Raw box in input pixels, not clipped."""
    b = [float(v) * box_scale for v in b]
    return [b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2] if fmt == "xywh" else b


def clip_box(b):
    """Clip to the image frame [0, IMGSZ], as Ultralytics does for its own boxes."""
    return [min(max(v, 0.0), float(IMGSZ)) for v in b]


mean_iou = {}
for fmt in ("xywh", "xyxy"):
    vals = [iou(clip_box(to_xyxy(a[a[:, 4 + rf["cls"]].argmax(), :4], fmt)), rf["xyxy"]) for a, rf in zip(anch, ref) if rf is not None]
    mean_iou[fmt] = float(np.mean(vals))
    print(f"  box format {fmt}: mean IoU (TFLite box clipped to the frame) with the PyTorch top box = {mean_iou[fmt]:.3f}")
BOX_FORMAT = max(mean_iou, key=mean_iou.get)
box_decided = mean_iou[BOX_FORMAT] > 0.8
BOX_FORMAT_TEXT = {"xywh": "centre x, centre y, width, height", "xyxy": "corners x1, y1, x2, y2"}[BOX_FORMAT]
print("DECIDED box format:", BOX_FORMAT_TEXT, "" if box_decided else "(NOT decided: IoU below 0.8)")


def top_det(a):
    s = a[:, 4:]
    k = int(s.max(1).argmax())
    c = int(s[k].argmax())
    raw_box = to_xyxy(a[k, :4], BOX_FORMAT)
    return c, float(s[k, c]), raw_box, clip_box(raw_box)


# 4) Side by side for 5 images, and agreement over all
print(f"\nTop detection, TFLite vs PyTorch, xyxy in {IMGSZ} x {IMGSZ} pixels. The TFLite box is clipped to [0, {IMGSZ}] before comparing, as Ultralytics clips its boxes. Agree = same class and every clipped coordinate within {AGREE_PX:.0f} px:")
diffs, agree = [], []
for i, (a, rf) in enumerate(zip(anch, ref)):
    if rf is None:
        continue
    c, sc, raw_bx, bx = top_det(a)
    d = max(abs(p - q) for p, q in zip(bx, rf["xyxy"]))
    diffs.append(d)
    agree.append(c == rf["cls"] and d <= AGREE_PX)
    if len(diffs) <= 5:
        print(f"  [{test[i][0]}]")
        print(f"    raw     {CLASS_NAMES[c]:8s} {sc:.3f} {[round(v, 1) for v in raw_bx]}")
        print(f"    clipped {CLASS_NAMES[c]:8s} {sc:.3f} {[round(v, 1) for v in bx]}")
        print(f"    PyTorch {CLASS_NAMES[rf['cls']]:8s} {rf['conf']:.3f} {[round(v, 1) for v in rf['xyxy']]}  -> max diff (clipped vs PyTorch) {d:.1f} px, {'AGREE' if agree[-1] else 'DISAGREE'}")
n_agree = int(sum(agree))
print(f"Agreement on {n_agree} of {len(agree)} images; max coordinate difference {max(diffs):.1f} px, mean {np.mean(diffs):.1f} px")
if n_agree < len(agree):
    print(f"WARNING: TFLite and PyTorch DISAGREE on {len(agree) - n_agree} of {len(agree)} images. Do not trust this file until that is explained.")

# 4b) Inverse mapping: TFLite box -> original image pixels with (x - pad_x) / gain, clipped AFTER the mapping,
# compared with PyTorch's own result on the unpadded original image (Ultralytics letterboxes and maps back itself).
orig_diffs = []
for a, t, ro in zip(anch, test, ref_orig):
    if ro is None:
        continue
    c, sc, raw_bx, bx = top_det(a)
    orig_diffs.append(max(abs(p - q) for p, q in zip(to_original(raw_bx, t[3]), ro["xyxy"])))
if orig_diffs:
    n_orig_ok = int(sum(d <= AGREE_PX for d in orig_diffs))
    print(f"Mapped back to the original image: {n_orig_ok} of {len(orig_diffs)} images within {AGREE_PX:.0f} px of PyTorch's own mapping (max {max(orig_diffs):.1f} px, mean {np.mean(orig_diffs):.1f} px).")
    print("  This second check also differs in padding: Ultralytics pads the original to a multiple of 32, our canvas is a fixed 640 x 640.")
print("Letterbox of the first image:", test[0][3])

# 5) Three overlays: ground truth blue, PyTorch green, TFLite red
overlays = []
for n in range(3):
    name, canvas, gt = test[n][:3]
    im = canvas.copy(); d = ImageDraw.Draw(im)
    for c, b in gt:
        d.rectangle(b, outline=(0, 90, 255), width=2)
    if ref[n]:
        d.rectangle(ref[n]["xyxy"], outline=(0, 200, 0), width=2)
    c, sc, raw_bx, bx = top_det(anch[n])
    d.rectangle(bx, outline=(255, 0, 0), width=2)  # clipped box
    d.text((4, 4), f"{CLASS_NAMES[c]} {sc:.2f} (red TFLite, green PyTorch, blue GT)", fill=(255, 255, 255))
    p = Path(DRIVE_ROOT, "exports", f"sanity_{'smoke_' if SMOKE else ''}{n + 1}.jpg")
    im.save(p, "JPEG", quality=90)
    overlays.append(str(p))
print("Saved", overlays)

CONVENTIONS = {
    "input": {"shape": in_shape, "dtype": np.dtype(inp["dtype"]).name, "layout": IN_LAYOUT, "channel_order": CHANNEL_ORDER,
              "pixel_range": PIXEL_RANGE, "decided_from_data": bool(input_decided),
              "evidence_mean_score_gap": {f"{c} {r}": round(v, 5) for (c, r), v in score_gap.items()}},
    "output": {"shape": out_shape, "dtype": np.dtype(out["dtype"]).name, "layout": OUT_LAYOUT, "anchors": n_anchors,
               "box_format": BOX_FORMAT_TEXT, "box_format_decided_from_data": bool(box_decided),
               "box_iou_evidence": {k: round(v, 4) for k, v in mean_iou.items()},
               "box_values": f"normalised (0 to 1), multiply by {IMGSZ}" if NORMALISED else f"pixels of the {IMGSZ} x {IMGSZ} input",
               "box_max_value_over_images": round(box_max, 3), "scores": SCORES,
               "score_min": round(s_min, 5), "score_max": round(s_max, 5)},
    "preprocessing": {
        "source": "Ultralytics 8.4.182 LetterBox (data/augment.py), the rule predict() uses; centre=True, pad 114, INTER_LINEAR are its defaults",
        "scale": f"{IMGSZ} / max(width, height)",
        "scale_up_allowed": True,
        "note_scale_up": "Ultralytics val uses scaleup=False. The two differ only for images whose longest side is below 640; the app should scale up.",
        "resized_size": "new_w = round(width * scale), new_h = round(height * scale)",
        "centred": True,
        "pad_x": f"round(({IMGSZ} - new_w) / 2 - 0.1)  (left padding, pixels)",
        "pad_y": f"round(({IMGSZ} - new_h) / 2 - 0.1)  (top padding, pixels)",
        "pad_value": 114,
        "interpolation": "cv2.INTER_LINEAR (bilinear)",
        "canvas": f"{IMGSZ} x {IMGSZ}, resized image pasted at (pad_x, pad_y)",
        "inverse_mapping": {
            "x_original": "(x - pad_x) / gain_x, with gain_x = new_w / width (about equal to scale)",
            "y_original": "(y - pad_y) / gain_y, with gain_y = new_h / height (about equal to scale)",
            "clip": "Clip to [0, width] x [0, height] AFTER this mapping, as Ultralytics scale_boxes does. Clipping to [0, 640] before the mapping is not enough: it keeps boxes that reach into the padding.",
        },
        "example_first_image": test[0][3],
    },
    "tflite_vs_pytorch": {"images_compared": len(agree), "agree": n_agree, "agree_px": AGREE_PX, "clipped_before_compare": True,
                          "max_diff_px": round(max(diffs), 2), "mean_diff_px": round(float(np.mean(diffs)), 2),
                          "mapped_back_to_original": ({"images": len(orig_diffs), "within_agree_px": n_orig_ok, "max_diff_px": round(max(orig_diffs), 2)} if orig_diffs else None),
                          "note": "detections are noise: random weights" if SMOKE else "trained weights"},
    "overlays": overlays,
}

## 12. Report
Writes a JSON report next to the model on Drive and prints it. It includes the conventions decided in cell 11.

In [ ]:
if "CONVENTIONS" not in globals():
    raise RuntimeError("Run cells 10 and 11 first (after a disconnect, run all again; finished steps are skipped).")

report = {
    "model_version": MODEL_VERSION,
    "run_mode": RUN_MODE,
    "weights": WEIGHTS_NOTE,
    "epochs_completed": epochs_done() if not SMOKE else 0,
    "epochs_planned": EPOCHS,
    "imgsz": IMGSZ,
    "class_order": CLASS_NAMES,
    "excluded_countries": EXCLUDE_COUNTRIES if not SMOKE else None,
    "settings_hash": SETTINGS_HASH if not SMOKE else None,
    "cache_file": TAR_PATH.name if not SMOKE else None,
    "split_file": SPLIT_FILE.name if not SMOKE else None,
    "map50_per_class": val_summary["map50_per_class"] if val_summary else None,
    "map50": val_summary["map50"] if val_summary else None,
    "map50_95": val_summary["map50_95"] if val_summary else None,
    "split_method": SPLIT_METHOD,
    "metric_note": "Measured on held-out blocks of RDD2022 train (blocks of 100 consecutive images per country, about 10%), not an official test split; the RDD2022 test split has no labels. No accuracy is claimed for Malaysian roads." if not SMOKE else "Smoke test: random weights, no metrics.",
    "train_counts": train_counts,
    "val_counts": val_counts,
    "tflite_path": str(TFLITE_PATH),
    "tflite_size_mb": round(TFLITE_SIZE_MB, 3),
    "tflite_input": {"shape": in_shape, "dtype": np.dtype(inp["dtype"]).name},
    "tflite_output": {"shape": out_shape, "dtype": np.dtype(out["dtype"]).name},
    "tflite_conventions": CONVENTIONS,
    "package_versions": PACKAGE_VERSIONS,
    "package_versions_before_export": CORE_VERSIONS,
}
report_path = Path(DRIVE_ROOT, "exports", f"report_{MODEL_VERSION}_{RUN_MODE}.json")
report_path.write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
print("Saved", report_path)